
# Pertemuan 3 - Data Cleaning

Nama : Abdul Zaki Al-Muttaqin
NIM : 240401010042
Kelas : IF403

## Tujuan Praktikum

Mempelajari teknik pembersihan data seperti menangani missing value, data duplikat, outlier, dan inkonsistensi data.

In [55]:
#Import library
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize

In [44]:
# Upload Dataset
from google.colab import files
uploaded = files.upload()

Saving housing_dirty.csv to housing_dirty (1).csv


In [45]:
# Load Dataset
df = pd.read_csv("housing_dirty.csv")
df.head()

,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang


In [46]:
# Eksplorasi Awal
print("Shape Dataset:")
print(df.shape)

print("\nInformasi Dataset:")
df.info()
df.describe()
df.isnull().sum()

Shape Dataset:
(130, 7)

Informasi Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB


,0
id,0
luas_m2,18
harga_juta,17
kota,0
kamar,10
tahun_bangun,0
kondisi,0


In [47]:
# Mengecek jumlah data duplikat
n_dup = df.duplicated().sum()
print("Jumlah data duplikat: ", n_dup)

# Menghapus data duplikat
df.drop_duplicates(inplace=True)

print("Shape setelah hapus duplikat:")
print(df.shape)

Jumlah data duplikat:  0
Shape setelah hapus duplikat:
(130, 7)


In [48]:
# Normalisasi kolom data
df['kota'] = df['kota'].str.strip().str.title()

# Normalisasi kolom kondisi
df['kondisi'] = df['kondisi'].str.strip().str.lower()

print("Normalisasi selesai")

print(df['kota'].unique())
print(df['kondisi'].unique())


Normalisasi selesai
['Jogja' 'Medan' 'Depok' 'Ygy' 'Jakarta' 'Yogyakarta' 'Bandung' 'Surabaya'
 'Dpk' 'Sby' 'Makassar' 'Mdn' 'Semarang' 'Smg' 'Bdg' 'Mksr']
['baik' 'bagus' 'sedang' 'baik sekali' 'rusak' 'cukup' 'perlu renovasi'
 'jelek']


In [49]:
#  Perbaiki Nama Kota
df['kota'] = df['kota'].replace({
    'Bdg': 'Bandung',
    'Dpk': 'Depok',
    'Mdn': 'Medan',
    'Smg': 'Semarang',
    'Sby': 'Surabaya',
    'Mksr': 'Makassar',
    'Ygy': 'Yogyakarta',
    'Jogja': 'Yogyakarta'
})

print(df['kota'].unique())

['Yogyakarta' 'Medan' 'Depok' 'Jakarta' 'Bandung' 'Surabaya' 'Makassar'
 'Semarang']


In [50]:
# imputasi missing values
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())

df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())

df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

print(df.isnull().sum())

id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64


In [51]:
# Menangani Outlier(IQR Fence)
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

print("Outlier berhasil ditangani")

Outlier berhasil ditangani


In [52]:
# Validasi Akhir
print("Total Missing Value:")
print(df.isnull().sum().sum())

print("\nTotal Duplikat:")
print(df.duplicated().sum())

Total Missing Value:
0

Total Duplikat:
0


In [53]:
# Simpan Dataset Bersih
df.to_csv("housing_clean.csv", index = False)
print("Dataset bersih berhasil disimpan")

Dataset bersih berhasil disimpan


In [54]:
# Akses API JSONPlaceholder
import requests
from pandas import json_normalize

url = "https://jsonplaceholder.typicode.com/users"

response = requests.get(url, timeout=10)

if response.status_code == 200:
    data = response.json()

    api_df = json_normalize(data)

    print(api_df[['id', 'name', 'email']])

else:
    print("Error:", response.status_code)

   id                      name                      email
0   1             Leanne Graham          Sincere@april.biz
1   2              Ervin Howell          Shanna@melissa.tv
2   3          Clementine Bauch         Nathan@yesenia.net
3   4          Patricia Lebsack  Julianne.OConner@kory.org
4   5          Chelsey Dietrich   Lucio_Hettinger@annie.ca
5   6      Mrs. Dennis Schulist    Karley_Dach@jasper.info
6   7           Kurtis Weissnat     Telly.Hoeger@billy.biz
7   8  Nicholas Runolfsdottir V       Sherwood@rosamond.me
8   9           Glenna Reichert    Chaim_McDermott@dana.io
9  10        Clementina DuBuque     Rey.Padberg@karina.biz


# Kesimpulan

Pada praktikum ini saya mempelajari proses data cleaning yang meliputi penanganan missing value, data duplikat, outlier, serta normalisasi data. Saya memahami bahwa data yang kotor dapat memengaruhi hasil analisis sehingga perlu dibersihkan terlebih dahulu.

Dari praktikum ini saya menyadari bahwa tahap data cleaning merupakan salah satu proses penting dalam Data Science sebelum data digunakan untuk analisis maupun pembuatan model.